## Week 4 (course alignment)

- Use frontier or OSS models to port Python to fast native code (C++ in day 3, Rust in day 5).
- Gather **system / toolchain** context (see `week4/system_info.py`) so models pick sane compiler flags.
- **Workflow:** emit `main.cpp` or `main.rs`, compile with aggressive optimizations, run and compare output.
- **Benchmarks in this notebook:** (1) Leibniz-style π partial sum from `week4/day3.ipynb`; (2) LCG + maximum subarray from `week4/day5.ipynb` (`python_hard`).
- **Runtime:** Set `WEEK4_QUICK=1` for a fast π check (10k iterations). The Python `python_hard` cell is Θ(n²) per seed (≈ minutes with n=10⁴); the Rust cell is Θ(n) and returns quickly. **Requires `rustc`** (rustup).

## Algorithms

1. **π benchmark:** finite alternating series (Madhava–Leibniz form) — **Θ(N)** additions of reciprocals; same evaluation order as Python so IEEE-754 `f64` output matches at 12 decimal places.
2. **LCG:** **Θ(1)** per value; 32-bit state with `(a·x + c) mod 2³²`, then affine map into `[min_val, max_val]`.
3. **Maximum subarray:** reference Python uses nested loops (**Θ(n²)** per run). **Kadane** computes the same maximum in **Θ(n)** per run; total over 20 seeds is **Θ(n)** instead of **Θ(n²)**. Streamed values avoid storing `n` integers.

In [ ]:
# Shared imports and Rust toolchain resolution
from __future__ import annotations

import io
import os
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

# Set WEEK4_QUICK=1 for a fast π sanity check (10_000 iterations). Default matches the course (200_000_000).
_QUICK = os.environ.get("WEEK4_QUICK", "").strip().lower() in ("1", "true", "yes")
PI_ITERATIONS = 10_000 if _QUICK else 200_000_000


def find_rustc() -> str:
    """Resolve rustc on PATH or the usual rustup location."""
    p = shutil.which("rustc")
    if p:
        return p
    candidate = Path.home() / ".cargo" / "bin" / "rustc"
    if candidate.is_file():
        return str(candidate)
    raise RuntimeError(
        "rustc not found. Install Rust from https://rustup.rs/ (or `curl https://sh.rustup.rs -sSf | sh`), "
        "then restart the kernel."
    )


def run_python_captured(code: str) -> str:
    """Execute Python source as in the course notebooks; return stdout."""
    buf = io.StringIO()
    old = sys.stdout
    sys.stdout = buf
    g: dict = {"__builtins__": __builtins__}
    try:
        exec(code, g, g)
    finally:
        sys.stdout = old
    return buf.getvalue()


def rust_compile_and_run(source: str) -> str:
    """Write main.rs to a temp dir, compile with release-style flags, run binary, return stdout."""
    rustc = find_rustc()
    exe_name = "main.exe" if sys.platform == "win32" else "main"
    with tempfile.TemporaryDirectory() as tmp:
        t = Path(tmp)
        rs_path = t / "main.rs"
        exe_path = t / exe_name
        rs_path.write_text(source, encoding="utf-8")
        cmd = [
            rustc,
            str(rs_path),
            "-C",
            "opt-level=3",
            "-C",
            "target-cpu=native",
            "-C",
            "codegen-units=1",
            "-C",
            "lto=fat",
            "-C",
            "panic=abort",
            "-o",
            str(exe_path),
        ]
        subprocess.run(cmd, check=True, text=True, capture_output=True)
        run = subprocess.run(
            [str(exe_path)],
            check=True,
            text=True,
            capture_output=True,
            cwd=t,
        )
        return run.stdout


def parse_pi_result(stdout: str) -> float:
    m = re.search(r"Result:\s*([0-9]+\.[0-9]+)", stdout)
    if not m:
        raise ValueError(f"Could not parse Result line from:\n{stdout!r}")
    return float(m.group(1))


def parse_subarray_total(stdout: str) -> int:
    m = re.search(r"Total Maximum Subarray Sum \(20 runs\):\s*(-?[0-9]+)", stdout)
    if not m:
        raise ValueError(f"Could not parse total from:\n{stdout!r}")
    return int(m.group(1))

In [ ]:
# π benchmark — Python reference (same logic as week4/day3.ipynb `pi` string)
pi_code = f"""import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations + 1):
        j = i * param1 - param2
        result -= (1 / j)
        j = i * param1 + param2
        result += (1 / j)
    return result

start_time = time.time()
result = calculate({PI_ITERATIONS}, 4, 1) * 4
end_time = time.time()

print(f"Result: {{result:.12f}}")
print(f"Execution Time: {{(end_time - start_time):.6f}} seconds")
"""

pi_py_out = run_python_captured(pi_code)
print(pi_py_out, end="")
pi_py_value = parse_pi_result(pi_py_out)

In [ ]:
# π benchmark — Rust (fill iteration count from Python cell)
PI_RUST_TEMPLATE = '// Mirrors Python: same loop bounds and -= / += reciprocal sequence (f64).\nfn calculate(iterations: u64, param1: f64, param2: f64) -> f64 {{\n    let mut result = 1.0_f64;\n    for i in 1..=iterations {{\n        let mut j = i as f64 * param1 - param2;\n        result -= 1.0 / j;\n        j = i as f64 * param1 + param2;\n        result += 1.0 / j;\n    }}\n    result\n}}\n\nfn main() {{\n    const ITERATIONS: u64 = {iterations};\n    let start = std::time::Instant::now();\n    let result = calculate(ITERATIONS, 4.0, 1.0) * 4.0;\n    let elapsed = start.elapsed().as_secs_f64();\n    println!("Result: {{:.12}}", result);\n    println!("Execution Time: {{:.6}} seconds", elapsed);\n}}\n'
pi_rust_src = PI_RUST_TEMPLATE.format(iterations=PI_ITERATIONS)
pi_rs_out = rust_compile_and_run(pi_rust_src)
print(pi_rs_out, end="")
pi_rs_value = parse_pi_result(pi_rs_out)
assert abs(pi_py_value - pi_rs_value) < 1e-12, (pi_py_value, pi_rs_value)
print("π: Python and Rust Result lines match (12 dp).")

In [ ]:
python_hard = """
# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))

"""

hard_py_out = run_python_captured(python_hard)
print(hard_py_out, end="")
hard_py_total = parse_subarray_total(hard_py_out)

In [ ]:
# LCG + max subarray — Rust (Kadane, streamed LCG)
HARD_RUST = '// LCG matches week4/day5 `python_hard`; max subarray via Kadane (O(n)), streaming O(1) extra memory.\nfn lcg_step(state: &mut u32) -> u32 {\n    const A: u64 = 1_664_525;\n    const C: u64 = 1_013_904_223;\n    let v = *state as u64;\n    let nxt = (A.wrapping_mul(v).wrapping_add(C)) % (1u64 << 32);\n    *state = nxt as u32;\n    *state\n}\n\nfn max_subarray_kadane_stream(mut inner: u32, n: usize, span: u32, min_val: i32) -> i64 {\n    let mut max_ending: i64 = 0;\n    let mut max_so_far: i64 = i64::MIN;\n    for _ in 0..n {\n        let raw = lcg_step(&mut inner);\n        let x = (raw % span) as i64 + min_val as i64;\n        max_ending = x.max(max_ending + x);\n        max_so_far = max_so_far.max(max_ending);\n    }\n    max_so_far\n}\n\nfn main() {\n    const N: usize = 10_000;\n    const INITIAL_SEED: u32 = 42;\n    const MIN_VAL: i32 = -10;\n    const MAX_VAL: i32 = 10;\n    let span = (MAX_VAL - MIN_VAL + 1) as u32;\n\n    let start = std::time::Instant::now();\n    let mut outer = INITIAL_SEED;\n    let mut total: i64 = 0;\n    for _ in 0..20 {\n        lcg_step(&mut outer);\n        let run_seed = outer;\n        total += max_subarray_kadane_stream(run_seed, N, span, MIN_VAL);\n    }\n    let elapsed = start.elapsed().as_secs_f64();\n    println!("Total Maximum Subarray Sum (20 runs): {}", total);\n    println!("Execution Time: {:.6} seconds", elapsed);\n}\n'
hard_rs_out = rust_compile_and_run(HARD_RUST)
print(hard_rs_out, end="")
hard_rs_total = parse_subarray_total(hard_rs_out)
assert hard_py_total == hard_rs_total, (hard_py_total, hard_rs_total)
print("Subarray totals: Python and Rust match exactly.")